# Tune retrieval pipeline for experiment setup section

Run Optuna studies for BM25, GTE, and BM25 + GTE RRF on the `gemma4` dataset.

In [ ]:
%cd ..

import optuna
import pandas as pd

from src.bm25 import Bm25Config
from src.dense_st import GteBaseConfig
from src.io_utils import read_json
from src.paths import RETRIEVAL_RUNS_DIR, SYNTHETIC_DATA_DIR
from src.pipeline import ExperimentConfig, run_experiment
from src.rrf_backend import RRFConfig
from src.span_selector import make_span_selector

tuning_query_path = SYNTHETIC_DATA_DIR / "gemma4" / "queries.jsonl"
tuning_qrels_path = SYNTHETIC_DATA_DIR / "gemma4" / "qrels.txt"
tuning_tag = "gemma4"

print(f"Tuning query path: {tuning_query_path}")
print(f"Tuning qrels path: {tuning_qrels_path}")

## Helpers

In [ ]:
optimization_metric = "ndcg@10"
secondary_metrics = [
    "recall@10",
    "mrr",
    "ndcg@100",
    "map",
    "precision@10",
]

DENSE_BATCH_SIZE = 4


def with_full_query_fallback(selector):
    def _select(text: str) -> list[str]:
        spans = selector(text)
        return spans if spans else [text]

    return _select


SPAN_SIZE_BY_MODE = {
    "sentence_1": 1,
    "sentence_2": 2,
    "sentence_3": 3,
    "sentence_4": 4,
    "sentence_5": 5,
    "sentence_7": 7,
    "sentence_10": 10,
}

SPAN_SELECTOR_BY_MODE = {"full_query": None}
SPAN_SELECTOR_BY_MODE.update(
    {
        mode: with_full_query_fallback(make_span_selector(span_size=span_size))
        for mode, span_size in SPAN_SIZE_BY_MODE.items()
    }
)


def build_span_selector(decomposition_mode: str):
    return SPAN_SELECTOR_BY_MODE[decomposition_mode]


def build_backend(backend_label: str, *, backend_rrf_k: int | None = None):
    if backend_label == "bm25":
        return Bm25Config()
    if backend_label == "dense":
        return GteBaseConfig(encode_batch_size=DENSE_BATCH_SIZE, score_batch_size=DENSE_BATCH_SIZE * 2)
    if backend_label == "bm25_gte_rrf":
        if backend_rrf_k is None:
            raise ValueError("backend_rrf_k is required for bm25_gte_rrf")
        return RRFConfig(
            children=(
                Bm25Config(),
                GteBaseConfig(encode_batch_size=DENSE_BATCH_SIZE, score_batch_size=DENSE_BATCH_SIZE * 2),
            ),
            rrf_k=backend_rrf_k,
        )
    raise ValueError(f"Unknown backend label: {backend_label}")


def build_experiment_name(
    *,
    prefix: str,
    backend_label: str,
    decomposition_mode: str,
    span_fusion_method: str,
    backend_rrf_k: int | None,
) -> str:
    backend_tag = backend_label
    if backend_label == "bm25_gte_rrf" and backend_rrf_k is not None:
        backend_tag = f"bm25-gte-rrf-k{backend_rrf_k}"
    parts = [prefix, backend_tag, decomposition_mode.replace("_", "-"), span_fusion_method.replace("_", "-")]
    return "-".join(parts)


def _normalize_optional_path(path):
    return None if path is None else str(path.resolve())


def _assert_same_dataset(*, existing_experiment, expected_query_path, expected_qrels_path, experiment_name):
    existing_query_path = existing_experiment.get("query_path")
    existing_qrels_path = existing_experiment.get("qrels_path")
    if existing_query_path != _normalize_optional_path(expected_query_path) or existing_qrels_path != _normalize_optional_path(expected_qrels_path):
        raise ValueError(
            f"Experiment '{experiment_name}' exists but uses a different dataset. "
            f"expected query_path={_normalize_optional_path(expected_query_path)} qrels_path={_normalize_optional_path(expected_qrels_path)}; "
            f"found query_path={existing_query_path} qrels_path={existing_qrels_path}"
        )


def run_or_load_experiment(
    experiment_name: str,
    *,
    query_path,
    qrels_path,
    backend,
    span_selector=None,
    span_fusion_method: str = "comb_sum",
):
    config = ExperimentConfig(
        experiment_name=experiment_name,
        query_path=query_path,
        qrels_path=qrels_path,
        backend=backend,
        span_selector=span_selector,
        span_fusion_method=span_fusion_method,
        span_score_normalization="none",
    )

    experiment_dir = RETRIEVAL_RUNS_DIR / experiment_name
    experiment_path = experiment_dir / "experiment.json"
    summary_path = experiment_dir / "eval" / "summary.json"
    if experiment_path.exists():
        existing_experiment = read_json(experiment_path)
        _assert_same_dataset(
            existing_experiment=existing_experiment,
            expected_query_path=config.query_path,
            expected_qrels_path=config.qrels_path,
            experiment_name=config.experiment_name,
        )
        if summary_path.exists():
            print(f"Skipping {experiment_name} (already exists)")
            return experiment_dir, read_json(summary_path)

    print(f"Running {experiment_name}")
    experiment_dir = run_experiment(config)
    return experiment_dir, read_json(experiment_dir / "eval" / "summary.json")


def make_objective(*, backend_label: str, search_space: dict, rows: list[dict]):
    def objective(trial: optuna.Trial) -> float:
        decomposition_mode = trial.suggest_categorical(
            "decomposition_mode",
            search_space["decomposition_mode"],
        )

        span_fusion_method = trial.suggest_categorical("span_fusion_method", search_space["span_fusion_method"])

        backend_rrf_k = None
        if backend_label == "bm25_gte_rrf":
            backend_rrf_k = trial.suggest_categorical("backend_rrf_k", search_space["backend_rrf_k"])

        span_selector = build_span_selector(decomposition_mode)
        backend = build_backend(backend_label, backend_rrf_k=backend_rrf_k)
        experiment_name = build_experiment_name(
            prefix=f"tune-setup-{tuning_tag}",
            backend_label=backend_label,
            decomposition_mode=decomposition_mode,
            span_fusion_method=span_fusion_method,
            backend_rrf_k=backend_rrf_k,
        )

        _, summary = run_or_load_experiment(
            experiment_name,
            query_path=tuning_query_path,
            qrels_path=tuning_qrels_path,
            backend=backend,
            span_selector=span_selector,
            span_fusion_method=span_fusion_method,
        )

        rows.append(
            {
                "trial_number": trial.number,
                "trial_key": f"{backend_label}__{trial.number}",
                "experiment_name": experiment_name,
                "backend_choice": backend_label,
                "decomposition_mode": decomposition_mode,
                "span_fusion_method": span_fusion_method,
                "backend_rrf_k": backend_rrf_k,
                **summary,
            }
        )
        return float(summary[optimization_metric])

    return objective


def finalize_results(rows: list[dict]) -> pd.DataFrame:
    if not rows:
        return pd.DataFrame()
    return (
        pd.DataFrame(rows)
        .sort_values(optimization_metric, ascending=False)
        .drop_duplicates(subset=["experiment_name"])
        .reset_index(drop=True)
    )

## Search spaces

In [ ]:
base_search_space = {
    "decomposition_mode": [
        "full_query",
        "sentence_1",
        "sentence_2",
        "sentence_3",
        "sentence_4",
        "sentence_5",
        "sentence_7",
        "sentence_10",
    ],
    "span_fusion_method": ["comb_sum", "comb_mnz"],
}

bm25_search_space = dict(base_search_space)
dense_search_space = dict(base_search_space)

hybrid_search_space = dict(base_search_space)
hybrid_search_space["backend_rrf_k"] = [1,2,3,4,5,6,7,8,9,10, 20, 30, 60, 90, 120, 180, 240]

n_trials = 100
study_seed = 42

print("BM25 search space:", bm25_search_space)
print("Dense search space:", dense_search_space)
print("Hybrid search space:", hybrid_search_space)

## BM25 study

In [ ]:
bm25_rows = []
bm25_study = optuna.create_study(
    study_name=f"setup-bm25-{tuning_tag}",
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=study_seed),
)
bm25_study.optimize(
    make_objective(
        backend_label="bm25",
        search_space=bm25_search_space,
        rows=bm25_rows,
    ),
    n_trials=n_trials,
)

bm25_results_df = finalize_results(bm25_rows)

print(f"Best BM25 trial: {bm25_study.best_trial.number}")
print(f"Best BM25 {optimization_metric}: {bm25_study.best_value:.4f}")
print(bm25_study.best_params)

## Dense (GTE) study

In [ ]:
dense_rows = []
dense_study = optuna.create_study(
    study_name=f"setup-dense-{tuning_tag}",
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=study_seed),
)
dense_study.optimize(
    make_objective(
        backend_label="dense",
        search_space=dense_search_space,
        rows=dense_rows,
    ),
    n_trials=n_trials,
)

dense_results_df = finalize_results(dense_rows)

print(f"Best dense trial: {dense_study.best_trial.number}")
print(f"Best dense {optimization_metric}: {dense_study.best_value:.4f}")
print(dense_study.best_params)

## BM25 + GTE RRF study

In [ ]:
hybrid_rows = []
hybrid_study = optuna.create_study(
    study_name=f"setup-hybrid-{tuning_tag}",
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=study_seed),
)
hybrid_study.optimize(
    make_objective(
        backend_label="bm25_gte_rrf",
        search_space=hybrid_search_space,
        rows=hybrid_rows,
    ),
    n_trials=n_trials,
)

hybrid_results_df = finalize_results(hybrid_rows)

print(f"Best hybrid trial: {hybrid_study.best_trial.number}")
print(f"Best hybrid {optimization_metric}: {hybrid_study.best_value:.4f}")
print(hybrid_study.best_params)

## Combined results

In [ ]:
all_results_df = (
    pd.concat([bm25_results_df, dense_results_df, hybrid_results_df], ignore_index=True, sort=False)
    .sort_values(optimization_metric, ascending=False)
    .reset_index(drop=True)
)

all_results_df[[
    "trial_number",
    "experiment_name",
    "backend_choice",
    "decomposition_mode",
    "span_fusion_method",
    "backend_rrf_k",
    optimization_metric,
    *secondary_metrics,
]].head(20)

## Candidate review

In [ ]:
candidate_columns = [
    "trial_key",
    "trial_number",
    "experiment_name",
    "backend_choice",
    "decomposition_mode",
    "span_fusion_method",
    "backend_rrf_k",
    optimization_metric,
    *secondary_metrics,
]

top_candidates_df = all_results_df[candidate_columns].head(10)
top_candidates_df

In [ ]:
selected_trial_number = int(top_candidates_df.iloc[0]["trial_number"])
selected_backend_choice = str(top_candidates_df.iloc[0]["backend_choice"])
selected_trial_df = all_results_df.loc[
    (all_results_df["trial_number"] == selected_trial_number)
    & (all_results_df["backend_choice"] == selected_backend_choice),
    candidate_columns,
]
selected_trial_df